In [25]:
import xarray as xr
import numpy as np
import xesmf as xe #this package is very unestable, if error try restart krenel
from dask.diagnostics import ProgressBar


In [ ]:
GFW_ds = xr.open_zarr(
    "./resources/GFW/GFW_AIS_Intl-FK_Trwl-Jgr_2012-2024.zarr",
    consolidated=True,
    decode_timedelta=True
)

In [27]:
GFW_ds

<xarray.Dataset> Size: 4GB
Dimensions:   (flag: 27, geartype: 2, time: 156, lat: 277, lon: 198)
Coordinates:
  * flag      (flag) object 216B 'ALB' 'ARE' 'ARG' 'BLZ' ... 'UKR' 'URY' 'VUT'
  * geartype  (geartype) object 16B 'SQUID_JIGGER' 'TRAWLERS'
  * lat       (lat) float64 2kB -60.0 -59.9 -59.8 -59.7 ... -32.6 -32.5 -32.4
  * lon       (lon) float64 2kB -69.61 -69.51 -69.41 ... -50.11 -50.01 -49.91
  * time      (time) datetime64[ns] 1kB 2012-01-01 2012-02-01 ... 2024-12-01
Data variables:
    hours     (time, lat, lon, flag, geartype) float64 4GB dask.array<chunksize=(12, 277, 198, 1, 1), meta=np.ndarray>
Attributes:
    created_by:    Ruben Barriuso
    date_created:  2025-12-29
    description:   Monthly fishing hours per grid cell in the SW Atlantic, \n...
    resolution:    0.1x0.1 degree
    source:        Global Fishing Watch (GFW)
    title:         SW Atlantic GFW AIS Fishing Effort Data (2012-2024)

In [ ]:
min_lon, min_lat, max_lon, max_lat = [-69.61, -60., -50., -32.45] #bbox of our study


res = 0.125
new_lons = np.arange(min_lon, max_lon + res, res)
new_lats = np.arange(min_lat, max_lat + res, res)

# Create target grid for from our study bbox with resolution of 0.125
ds_tgt = xr.Dataset({
    'lat': (['lat'], new_lats),
    'lon': (['lon'], new_lons)})

regridder = xe.Regridder(GFW_ds, ds_tgt, 'conservative') #regrid from the 0.1 resolution to the 0.125 resolution of our study 
with ProgressBar():
    GFW_regridded = regridder(GFW_ds).compute() #compute (as is a dask zarray)


[########################################] | 100% Completed | 7.99 sms


In [ ]:
#comparision of the original and the regrided grids
print("original size", GFW_ds.sizes)
print("regrided size", GFW_regridded.sizes)
lat_diff = np.diff(GFW_regridded.lat)
lon_diff = np.diff(GFW_regridded.lon)
print("Regrided spacing:")
print("Lat spacing:", np.unique(lat_diff), "Lon spacing:", np.unique(lon_diff))
print("Original bounds:")
print(GFW_ds.lat.min().item(), GFW_ds.lat.max().item())
print(GFW_ds.lon.min().item(), GFW_ds.lon.max().item())
print("Regrided bounds:")
print(GFW_regridded.lat.min().item(), GFW_regridded.lat.max().item())
print(GFW_regridded.lon.min().item(), GFW_regridded.lon.max().item())

original size Frozen({'flag': 27, 'geartype': 2, 'time': 156, 'lat': 277, 'lon': 198})
regrided size Frozen({'time': 156, 'flag': 27, 'geartype': 2, 'lat': 222, 'lon': 158})
Regrided spacing:
Lat spacing: [0.125] Lon spacing: [0.125]
Original bounds:
-60.0 -32.39999999999961
-69.61 -49.91000000000112
Regrided bounds:
-60.0 -32.375
-69.61 -49.985


In [30]:
#Adding metadata and description to this zarr dataset
GFW_regridded.attrs['title'] = "Regrided SW Atlantic GFW AIS Fishing Effort Data (2012-2024)"
GFW_regridded.attrs["description"] = (
    """Monthly fishing hours per grid cell in the SW Atlantic, 
    aligned to 0.125° lat/lon grid
    only data from trawlers and squid jiggers vessels included,
    only data within the available fishing area is included which comprises
    international waters and claimed Falkland Islands EEZ."""
)

GFW_regridded.attrs["source"] = "AIS-based fishing effort dataset"
GFW_regridded.attrs["created_by"] = "Ruben Barriuso"
GFW_regridded.attrs["resolution"] = "0.125x0.125 degree"
GFW_regridded.attrs["source"] = "Global Fishing Watch (GFW)"

GFW_regridded["hours"].attrs["long_name"] = "Fishing effort in AIS hours"
GFW_regridded["hours"].attrs["description"] = (
    "Total monthly fishing effort observed in each grid cell from GFW"
)

In [ ]:
for var in GFW_regridded.variables:
    GFW_regridded[var].encoding = {} #this is for elimiating the encoding, causes problems in future analysis
GFW_regridded.to_zarr("./data/processed/GFW_AIS.zarr", 
                      mode='w',
                      consolidated=True)#creates a metadata file for faster access)